In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

import os, glob
vllm_wheels = glob.glob('/kaggle/input/vllm-gptoss-wheel/*.whl')
if vllm_wheels:
    os.system(f'pip install {vllm_wheels[0]} --quiet')
else:
    os.system(
        'pip install --pre "vllm==0.10.1+gptoss" '
        '--extra-index-url https://wheels.vllm.ai/gpt-oss/ '
        '--extra-index-url https://download.pytorch.org/whl/nightly/cu128 '
        '--index-strategy unsafe-best-match --quiet'
    )
os.system('pip install openai-harmony transformers kernels --quiet')

In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# ARC-AGI-3: GPT-OSS 120B + Acontext-Inspired Skill Memory
#
# Offline implementation của Acontext skill memory pattern:
#   - Skill files: plain Markdown, per-game + shared
#   - Distillation: LLM pass sau mỗi level để update skills
#   - Cross-game: patterns persist và tái sử dụng
#   - No server / no embeddings / no internet needed
# =====================================================================

import json
import logging
import os
import random
import re
import time
import traceback
from collections import deque
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState


# ─────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────

SKILL_DIR     = Path("/kaggle/working/skills")
SHARED_DIR    = SKILL_DIR / "shared"
COLOR_CHARS   = "0123456789ABCDEF"

MODEL_PATHS = [
    "/kaggle/input/gpt-oss-120b",
    "/kaggle/input/gpt-oss-120b-weights",
    "/kaggle/input/openai-gpt-oss-120b",
    "openai/gpt-oss-120b",
]

ACTION_MAP = {
    'ACTION1': GameAction.ACTION1, 'ACTION2': GameAction.ACTION2,
    'ACTION3': GameAction.ACTION3, 'ACTION4': GameAction.ACTION4,
    'ACTION5': GameAction.ACTION5, 'ACTION6': GameAction.ACTION6,
    'RESET':   GameAction.RESET,
}


# ─────────────────────────────────────────────────────────
# LLM Engine (singleton)
# ─────────────────────────────────────────────────────────

_ENGINE    = None
_TOKENIZER = None
_USE_VLLM  = False


def _find_model() -> str:
    for p in MODEL_PATHS:
        if os.path.exists(p):
            print(f"[LLM] Model found: {p}")
            return p
    return MODEL_PATHS[-1]


def init_engine():
    global _ENGINE, _TOKENIZER, _USE_VLLM
    if _ENGINE is not None:
        return

    model_path = _find_model()
    try:
        from vllm import LLM
        from transformers import AutoTokenizer
        print("[LLM] Initializing vLLM...")
        _ENGINE = LLM(
            model=model_path,
            dtype="auto",
            max_model_len=8192,
            gpu_memory_utilization=0.88,
            max_num_seqs=4,
            trust_remote_code=True,
        )
        _TOKENIZER = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        _USE_VLLM  = True
        print("[LLM] vLLM ready.")
    except Exception as e:
        print(f"[LLM] vLLM failed ({e}), using transformers...")
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
        _TOKENIZER = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.bfloat16,
            device_map="auto", trust_remote_code=True
        )
        _ENGINE   = pipeline("text-generation", model=mdl, tokenizer=_TOKENIZER)
        _USE_VLLM = False


def llm_call(
    messages: List[Dict],
    reasoning: str = "low",
    max_tokens: int = 512,
    stop: Optional[List[str]] = None,
) -> str:
    """
    GPT-OSS 120B inference via vLLM (hoặc transformers fallback).
    reasoning level điều chỉnh depth của chain-of-thought.
    """
    global _ENGINE, _TOKENIZER, _USE_VLLM
    if _ENGINE is None:
        init_engine()

    # Inject reasoning level vào system message
    msgs = list(messages)
    if msgs and msgs[0]['role'] == 'system':
        msgs[0] = dict(msgs[0])
        if 'Reasoning:' not in msgs[0]['content']:
            msgs[0]['content'] += f"\nReasoning: {reasoning}"

    try:
        prompt = _TOKENIZER.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in msgs) + "\nASSISTANT:"

    _stop = (stop or []) + ["<|im_end|>", "</response>"]

    if _USE_VLLM:
        from vllm import SamplingParams
        temp = {'low': 0.2, 'medium': 0.4, 'high': 0.6}.get(reasoning, 0.3)
        out  = _ENGINE.generate([prompt], SamplingParams(
            temperature=temp, max_tokens=max_tokens, stop=_stop
        ))
        raw = out[0].outputs[0].text
    else:
        out = _ENGINE(prompt, max_new_tokens=max_tokens, do_sample=(reasoning != 'low'),
                      temperature=0.4, return_full_text=False)
        raw = out[0]['generated_text']

    # Strip reasoning block
    clean = re.sub(r'<reasoning>.*?</reasoning>', '', raw, flags=re.DOTALL).strip()
    return clean or raw.strip()


# ─────────────────────────────────────────────────────────
# Acontext-Inspired Skill Memory
# ─────────────────────────────────────────────────────────

class SkillMemory:
    """
    Offline implementation của Acontext skill memory pattern.

    Acontext concepts được implement:
    - Skill files: Markdown files per game (game_mechanic, failure_log, strategy)
    - Distillation: LLM pass extract what worked/failed → update skills
    - Space: shared/ folder là cross-game 'space'
    - Progressive disclosure: inject chỉ relevant sections vào context

    Không implement:
    - Acontext server / API (offline)
    - Embedding search (dùng full-text injection thay)
    - Task Agent background process (synchronous thay)
    """

    MECHANIC_TEMPLATE = """# Game Mechanic: {game_id}

## Action Effects
<!-- Điền khi agent phát hiện ra action nào làm gì -->
- ACTION1: unknown
- ACTION2: unknown
- ACTION3: unknown
- ACTION4: unknown
- ACTION5: unknown
- ACTION6 (x,y): unknown

## Win Condition
unknown

## Discovered Patterns
none yet
"""

    FAILURE_TEMPLATE = """# Failure Log: {game_id}

## Failed Sequences
none yet

## Dead-End States
none yet

## Actions With No Effect
none yet
"""

    STRATEGY_TEMPLATE = """# Strategy: {game_id} — Level {level}

## Current Hypothesis
unknown — explore first

## Next Actions To Try
1. Explore ACTION1-ACTION5 to observe effects
2. Try ACTION6 on colored/distinct cells

## Progress
Level {level} — {actions} actions taken
"""

    SHARED_TEMPLATE = """# Cross-Game Pattern Library

## Recurring Mechanics
none yet

## Reliable Exploration Strategies
- Always try ACTION1-ACTION5 first on a new level to observe effects
- ACTION6 on cells that stand out (different color, isolated) often triggers interactions
- If stuck 6+ actions: try the reverse of what seemed to be working

## Anti-Patterns (avoid)
- Repeating the same action 3+ times with no frame change
"""

    def __init__(self, game_id: str):
        self.game_id  = game_id
        self.game_dir = SKILL_DIR / game_id
        self.game_dir.mkdir(parents=True, exist_ok=True)
        SHARED_DIR.mkdir(parents=True, exist_ok=True)

        # Init skill files if not exist
        self._init_file('game_mechanic.md', self.MECHANIC_TEMPLATE.format(game_id=game_id))
        self._init_file('failure_log.md',   self.FAILURE_TEMPLATE.format(game_id=game_id))
        self._init_file('strategy.md',      self.STRATEGY_TEMPLATE.format(game_id=game_id, level=0, actions=0))

        shared_path = SHARED_DIR / 'cross_game_patterns.md'
        if not shared_path.exists():
            shared_path.write_text(self.SHARED_TEMPLATE)

    def _init_file(self, name: str, content: str):
        p = self.game_dir / name
        if not p.exists():
            p.write_text(content)

    def read(self, name: str) -> str:
        return (self.game_dir / name).read_text()

    def write(self, name: str, content: str):
        (self.game_dir / name).write_text(content)

    def read_shared(self) -> str:
        return (SHARED_DIR / 'cross_game_patterns.md').read_text()

    def write_shared(self, content: str):
        (SHARED_DIR / 'cross_game_patterns.md').write_text(content)

    def build_context(self, level: int, actions_taken: int) -> str:
        """
        Acontext 'progressive disclosure': inject chỉ những gì agent cần
        vào LLM context. Không dump toàn bộ history.

        Early level → cần mechanic + shared patterns
        Later level → cần strategy + failure log để avoid mistakes
        """
        parts = []

        # Luôn inject cross-game patterns (compact)
        shared = self.read_shared()
        parts.append(f"=== CROSS-GAME KNOWLEDGE ===\n{shared}")

        # Game mechanic (chỉ nếu không còn 'unknown')
        mechanic = self.read('game_mechanic.md')
        if 'unknown' not in mechanic or level > 0:
            parts.append(f"=== GAME MECHANIC ===\n{mechanic}")

        # Strategy (luôn inject)
        strategy = self.read('strategy.md')
        parts.append(f"=== CURRENT STRATEGY ===\n{strategy}")

        # Failure log (chỉ inject nếu có failures)
        failures = self.read('failure_log.md')
        if 'none yet' not in failures:
            # Chỉ lấy phần đầu để không overflow context
            parts.append(f"=== FAILURES TO AVOID ===\n{failures[:600]}")

        return "\n\n".join(parts)

    def distill(
        self,
        episode: List[Dict],   # list of {frame_text, action, diff, reward}
        level: int,
        win: bool,
    ):
        """
        Acontext 'Distillation': LLM pass nhìn lại episode,
        extract learnings, update skill files.

        Chạy sau mỗi level complete hoặc game over.
        """
        if len(episode) < 3:
            return  # quá ít để học

        # Tóm tắt episode (chỉ lấy keyframes để tránh overflow)
        keyframes = episode[:3] + episode[-3:]  # first 3 + last 3
        episode_summary = "\n".join(
            f"Action {i+1}: {e['action']} → {e['diff']}"
            for i, e in enumerate(keyframes)
        )

        current_mechanic  = self.read('game_mechanic.md')
        current_failures  = self.read('failure_log.md')
        current_strategy  = self.read('strategy.md')

        prompt_msgs = [
            {"role": "system", "content": (
                "You are a skill extractor for an ARC-AGI-3 game agent. "
                "Analyze the agent's episode and update the skill files. "
                "Be concise. Extract only concrete, reusable learnings. "
                "Reasoning: medium"
            )},
            {"role": "user", "content": f"""Game: {self.game_id} | Level: {level} | Outcome: {'WIN' if win else 'FAIL'}
Total actions: {len(episode)}

EPISODE SUMMARY (keyframes):
{episode_summary}

CURRENT SKILL FILES:
--- game_mechanic.md ---
{current_mechanic[:800]}

--- failure_log.md ---
{current_failures[:600]}

--- strategy.md ---
{current_strategy[:600]}

Based on this episode, output updated versions of the three skill files.
Format exactly:
<mechanic>
[full updated game_mechanic.md content]
</mechanic>
<failures>
[full updated failure_log.md content]
</failures>
<strategy>
[full updated strategy.md content for next level]
</strategy>"""},
        ]

        raw = llm_call(prompt_msgs, reasoning='medium', max_tokens=1024,
                       stop=['</strategy>'])
        raw += '</strategy>'  # ensure closing tag if truncated

        # Parse và update files
        mechanic_m = re.search(r'<mechanic>(.*?)</mechanic>', raw, re.DOTALL)
        failures_m = re.search(r'<failures>(.*?)</failures>', raw, re.DOTALL)
        strategy_m = re.search(r'<strategy>(.*?)</strategy>', raw, re.DOTALL)

        if mechanic_m:
            self.write('game_mechanic.md', mechanic_m.group(1).strip())
        if failures_m:
            self.write('failure_log.md', failures_m.group(1).strip())
        if strategy_m:
            self.write('strategy.md', strategy_m.group(1).strip())

        print(f"[Skill] Distillation complete for {self.game_id} L{level} ({'WIN' if win else 'FAIL'})")

    def update_shared(self, new_pattern: str):
        """
        Acontext 'Space': add newly discovered cross-game pattern
        vào shared skill library.
        """
        current = self.read_shared()
        if new_pattern.strip() and new_pattern not in current:
            updated = current + f"\n- {new_pattern.strip()}"
            self.write_shared(updated)


# ─────────────────────────────────────────────────────────
# Frame utilities
# ─────────────────────────────────────────────────────────

def grid_to_ascii(grid: np.ndarray, max_size: int = 28) -> str:
    H, W = grid.shape
    g = grid[:max_size, :max_size]
    H2, W2 = g.shape
    col_hdr = "    " + " ".join(f"{c%10}" for c in range(W2))
    rows = [col_hdr]
    for r in range(H2):
        row = " ".join(COLOR_CHARS[int(v)] if 0 <= int(v) <= 15 else '?' for v in g[r])
        rows.append(f"{r:3d}| {row}")
    if H > max_size or W > max_size:
        rows.append(f"[truncated {H}×{W}]")
    return "\n".join(rows)


def frame_to_text(frame: FrameData) -> Tuple[str, np.ndarray]:
    raw = np.array(frame.frame, dtype=np.int64)
    if raw.ndim == 3:
        raw = raw[-1]
    state = frame.state.name if hasattr(frame.state, 'name') else str(frame.state)
    level = getattr(frame, 'levels_completed', 0)
    return f"State={state} Level={level} Grid={raw.shape[0]}×{raw.shape[1]}\n{grid_to_ascii(raw)}", raw


def summarize_diff(a: Optional[np.ndarray], b: np.ndarray) -> str:
    if a is None or a.shape != b.shape:
        return "(first frame)"
    diff = np.argwhere(a != b)
    if len(diff) == 0:
        return "NO CHANGE"
    n = len(diff)
    if n <= 5:
        return ", ".join(f"({r},{c}):{int(a[r,c])}→{int(b[r,c])}" for r,c in diff)
    return f"{n} cells changed rows {diff[:,0].min()}-{diff[:,0].max()}"


def parse_action(text: str) -> Tuple[Optional[str], Optional[Tuple[int,int]]]:
    m = re.search(r'<action>\s*(.+?)\s*</action>', text, re.IGNORECASE | re.DOTALL)
    raw = m.group(1).strip() if m else text.strip()
    m6 = re.search(r'ACTION6\s+(\d+)[,\s]+(\d+)', raw, re.IGNORECASE)
    if m6:
        return 'ACTION6', (int(m6.group(1)), int(m6.group(2)))
    for a in ['ACTION1','ACTION2','ACTION3','ACTION4','ACTION5','RESET']:
        if a.lower() in raw.lower():
            return a, None
    n = re.search(r'\b([1-5])\b', raw)
    if n:
        return f'ACTION{n.group(1)}', None
    return None, None


# ─────────────────────────────────────────────────────────
# System prompt
# ─────────────────────────────────────────────────────────

SYSTEM_PROMPT = """\
You are an expert ARC-AGI-3 game agent. You will be given:
1. Skill memory from previous episodes (what you already know)
2. Current game frame (grid state)

GRID: Each character = one cell color (0-9, A-F). Row/col indices shown.

ACTIONS:
- ACTION1-ACTION5: simple interactions (move, rotate, interact — learn by observing)
- ACTION6 x,y: click at column x, row y (0-indexed from top-left)
- RESET: only when state=GAME_OVER

GOAL: Complete levels with minimum actions. Use skill memory to avoid repeating mistakes.
If skill memory says an action has no effect → don't repeat it.
If skill memory has a working strategy → follow it.

RESPOND:
One line of reasoning, then:
<action>ACTION_NAME</action>  or  <action>ACTION6 x,y</action>"""


def build_messages(
    game_id: str,
    skill_context: str,
    frame_text: str,
    diff: str,
    prev_action: Optional[str],
    available,
    conversation: List[Dict],
) -> List[Dict]:
    avail = ", ".join(
        (a.name if hasattr(a, 'name') else f'ACTION{a}') for a in (available or [])
    ) or "ACTION1-ACTION6"

    # Skill memory trong system context (Acontext: progressive disclosure)
    sys_with_skills = SYSTEM_PROMPT + f"\n\n{skill_context}"

    diff_line = f"\n→ {prev_action} caused: {diff}" if prev_action else ""
    user_msg  = f"[{game_id}] available={avail}\n{frame_text}{diff_line}\nChoose:"

    msgs = [{"role": "system", "content": sys_with_skills}]
    for turn in conversation[-3:]:
        msgs.append({"role": "assistant", "content": turn['response']})
        msgs.append({"role": "user",      "content": turn['frame_summary']})
    msgs.append({"role": "user", "content": user_msg})
    return msgs


# ─────────────────────────────────────────────────────────
# Agent
# ─────────────────────────────────────────────────────────

class MyAgent(Agent):
    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 8

    # Distillation triggers
    DISTILL_ON_WIN       = True
    DISTILL_ON_STUCK     = 20    # distill sau N actions stuck liên tiếp
    DISTILL_EVERY        = 50    # distill định kỳ mỗi N actions trong level

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.start_time = time.time()

        # Logging
        log_dir = Path("/kaggle/working/agent_logs")
        log_dir.mkdir(exist_ok=True)
        self.logger = logging.getLogger(self.game_id)
        if not self.logger.handlers:
            fh = logging.FileHandler(log_dir / f"{self.game_id}.log")
            fh.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
            self.logger.addHandler(fh)
            self.logger.setLevel(logging.INFO)

        # Skill memory
        self.skills = SkillMemory(self.game_id)

        # Per-level state
        self.current_level    = -1
        self.prev_grid        = None
        self.prev_action_name = None
        self.stuck_count      = 0
        self.conversation:    List[Dict] = []
        self.episode:         List[Dict] = []  # for distillation

        # Stats
        self.n_llm  = 0
        self.n_dist = 0

        try:
            init_engine()
        except Exception as e:
            self.logger.warning(f"Engine prewarm failed: {e}")

    # ── Lifecycle ───────────────────────────────────────────

    def append_frame(self, frame: FrameData) -> None:
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if frame.guid:
            self.guid = frame.guid
        if hasattr(self, 'recorder') and not self.is_playback:
            self.recorder.record(json.loads(frame.model_dump_json()))

    def is_done(self, frames, latest_frame):
        try:
            elapsed = time.time() - self.start_time
            if latest_frame.state is GameState.WIN:
                # Distill on win
                self._maybe_distill(win=True)
                return True
            return elapsed >= 8 * 3600 - 5 * 60
        except Exception:
            return True

    # ── Helpers ─────────────────────────────────────────────

    def _get_level(self, frame):
        return getattr(frame, 'score', None) or getattr(frame, 'levels_completed', 0)

    def _reset_level(self):
        """Reset per-level state. Skills survive across levels."""
        self.prev_grid        = None
        self.prev_action_name = None
        self.stuck_count      = 0
        self.conversation     = []
        self.episode          = []

    def _make_action(self, name: str, coords=None, note: str = '') -> GameAction:
        act = ACTION_MAP.get(name.upper(), GameAction.ACTION1)
        act.reasoning = note
        if name.upper() == 'ACTION6' and coords:
            act.set_data({'x': int(coords[0]), 'y': int(coords[1])})
        return act

    def _maybe_distill(self, win: bool = False):
        """Acontext distillation: LLM pass để update skill files."""
        if len(self.episode) < 5:
            return
        t0 = time.time()
        self.skills.distill(self.episode, self.current_level, win)
        self.n_dist += 1
        # Reset episode sau distillation
        self.episode = []
        self.logger.info(f"Distill #{self.n_dist} t={time.time()-t0:.1f}s win={win}")

    # ── Core ────────────────────────────────────────────────

    def choose_action(self, frames, latest_frame):
        try:
            return self._impl(frames, latest_frame)
        except Exception as e:
            traceback.print_exc()
            a = GameAction.ACTION1
            a.reasoning = f'crash: {e}'
            return a

    def _impl(self, frames, latest_frame):
        # ── Level change ──
        level = self._get_level(latest_frame)
        if level != self.current_level:
            if self.episode:
                self._maybe_distill(win=False)  # distill trước khi reset
            print(f"[{self.game_id}] Level {self.current_level}→{level} llm={self.n_llm} dist={self.n_dist}")
            self._reset_level()
            self.current_level = level

        # ── Terminal ──
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            if self.episode:
                self._maybe_distill(win=False)
            self._reset_level()
            return self._make_action('RESET', note='terminal')

        # ── Frame + diff ──
        frame_text, curr_grid = frame_to_text(latest_frame)
        diff = summarize_diff(self.prev_grid, curr_grid)

        # ── Record episode step ──
        if self.prev_action_name:
            changed = diff != "NO CHANGE" and diff != "(first frame)"
            self.stuck_count = 0 if changed else self.stuck_count + 1
            self.episode.append({
                'frame_text': frame_text[:200],
                'action':     self.prev_action_name,
                'diff':       diff,
                'reward':     1.0 if changed else 0.0,
            })

        self.prev_grid = curr_grid

        # ── Periodic distillation ──
        if self.action_counter > 0 and self.action_counter % self.DISTILL_EVERY == 0:
            self._maybe_distill(win=False)
        elif self.stuck_count > 0 and self.stuck_count % self.DISTILL_ON_STUCK == 0:
            self._maybe_distill(win=False)

        # ── Skill context (Acontext progressive disclosure) ──
        skill_context = self.skills.build_context(level, self.action_counter)

        # ── Reasoning level based on situation ──
        if self.action_counter == 0 or level != self.current_level:
            rlevel  = 'medium'   # new level: understand game
            max_tok = 300
        elif self.stuck_count >= 10:
            rlevel  = 'high'     # stuck hard: think deeply
            max_tok = 512
        elif self.stuck_count >= 4:
            rlevel  = 'medium'   # stuck a bit: reconsider
            max_tok = 256
        else:
            rlevel  = 'low'      # normal: fast action
            max_tok = 128

        available = getattr(latest_frame, 'available_actions', None)
        msgs = build_messages(
            self.game_id, skill_context, frame_text,
            diff, self.prev_action_name, available, self.conversation
        )

        # ── LLM call ──
        t0  = time.time()
        raw = llm_call(msgs, reasoning=rlevel, max_tokens=max_tok)
        dt  = time.time() - t0
        self.n_llm += 1

        self.logger.info(f"a#{self.action_counter} r={rlevel} t={dt:.2f}s stuck={self.stuck_count}")

        # ── Parse ──
        action_name, coords = parse_action(raw)
        if action_name is None:
            action_name = random.choice(['ACTION1','ACTION2','ACTION3','ACTION4','ACTION5'])
            note = f'parse_fail→{action_name}'
        else:
            note = f'llm({rlevel},{dt:.1f}s)'

        # Save conversation turn (compact)
        self.conversation.append({
            'frame_summary': frame_text[:150],
            'response':      raw,
        })

        self.prev_action_name = action_name
        return self._make_action(action_name, coords, note)


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('OK')
    submission.head()